# LeetCode 590 — N-ary Tree Postorder Traversal bằng Python

**Link đề:** https://leetcode.com/problems/n-ary-tree-postorder-traversal/
**Độ khó:** Easy · **Chủ đề:** Stack, Tree, Depth-First Search

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val: Optional[int] = None, children: Optional[List['Node']] = None):
        self.val = val
        self.children = children
"""

class Solution:
    def postorder(self, root: 'Node') -> List[int]:
        ...
```

## Mục tiêu học
1. Hiểu **cây N-ary** và cách LeetCode ghi cây thành list với `null` làm dấu ngăn nhóm con.
2. Hiểu **duyệt hậu thứ tự (postorder)**: duyệt hết các cây con (trái → phải) rồi mới ghi node cha.
3. Viết lời giải **đệ quy** và theo dõi được thời điểm "ghi" trên call stack.
4. Trả lời Follow-up bằng **3 cách lặp**: (a) mẹo "preorder ngược rồi đảo list", (b) stack có **cờ đã thăm**, (c) stack `(node, chỉ số con)` mô phỏng đệ quy.
5. Trace từng bước, chứng minh invariant, phân tích Time/Space, làm 6 bài tập có đáp án cuối notebook.

## 1. Đề bài và ví dụ

Cho `root` của một **cây N-ary**, trả về list giá trị các node theo thứ tự **postorder**.

Ví dụ 1: `root = [1,null,3,2,4,null,5,6]` → `[5,6,3,2,4,1]`

```text
            1
         /  |  \
        3   2   4
       / \
      5   6
```

Postorder = "xong hết con thì mới ghi cha": vào `3` → xong `5`, `6` → ghi `3` → ghi `2` → ghi `4` → cuối cùng ghi `1`.

Ví dụ 2: `root = [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]`
→ `[2,6,14,11,7,3,12,8,4,13,9,10,5,1]`

**Constraints:**
- Số node trong `[0, 10^4]` (có thể rỗng → `[]`).
- `0 <= Node.val <= 10^4`.
- Chiều cao cây `<= 1000`.

**Follow up:** đệ quy thì dễ, bạn có làm được bằng vòng lặp không?

Nhận xét đầu tiên: **root luôn đứng cuối cùng** trong postorder (ngược với preorder – root đứng đầu).

## 2. Nền tảng: `class Node`, dựng cây từ list LeetCode, in cây

### 2.1. Node N-ary
Mỗi node có `val` và `children` – một **list** con theo thứ tự trái → phải. Node lá có `children` rỗng. Ta tự định nghĩa `Node` (dùng `children or []` để luôn có list).

### 2.2. Cách LeetCode ghi cây
Ghi theo tầng; sau root là một `null`, sau đó là **nhóm con** của từng node (theo thứ tự node xuất hiện), các nhóm ngăn nhau bằng `null`:

```text
[1, null, 3, 2, 4, null, 5, 6]
  root |  con của 1 | con của 3
```

Để dựng lại, ta dùng **queue** `deque`: node nào được tạo trước thì "nhận nhóm con" trước (FIFO – vào trước ra trước).

In [1]:
from collections import deque
from typing import List, Optional
import random, sys


class Node:
    def __init__(self, val: Optional[int] = None, children: Optional[List['Node']] = None):
        self.val = val
        self.children = children if children is not None else []

    def __repr__(self):
        return f"Node({self.val})"


def build_nary(values):
    """Dựng cây N-ary từ list kiểu LeetCode (None ngăn các nhóm con)."""
    if not values:
        return None
    root = Node(values[0])
    queue = deque([root])
    i, n = 1, len(values)
    while queue and i < n:
        i += 1                          # bỏ dấu None mở đầu nhóm
        parent = queue.popleft()
        while i < n and values[i] is not None:
            child = Node(values[i])
            parent.children.append(child)
            queue.append(child)
            i += 1
    return root


def show_nary(root, depth=0):
    if root is None:
        print("(cây rỗng)")
        return
    print("    " * depth + str(root.val))
    for child in root.children:
        show_nary(child, depth + 1)


EX1 = [1, None, 3, 2, 4, None, 5, 6]
EX2 = [1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14]
show_nary(build_nary(EX1))

1
    3
        5
        6
    2
    4


### 2.3. Thử `deque` và `list` – hai công cụ chính của bài

- `deque.popleft()` lấy phần tử **đầu** (queue, FIFO) – dùng để dựng cây.
- `list.pop()` lấy phần tử **cuối** (stack, LIFO) – dùng cho DFS lặp.
- `list.reverse()` đảo **tại chỗ** (không trả về gì); `lst[::-1]` tạo list mới đã đảo.

In [2]:
q = deque([1, 2, 3])
print("queue popleft:", q.popleft(), "| còn", list(q))

s = [1, 2, 3]
print("stack pop    :", s.pop(), "| còn", s)

a = [1, 2, 3]
r = a.reverse()
print("a.reverse() trả về:", r, "| a giờ là:", a)
print("[1, 2, 3][::-1] =", [1, 2, 3][::-1])

queue popleft: 1 | còn [2, 3]
stack pop    : 3 | còn [1, 2]
a.reverse() trả về: None | a giờ là: [3, 2, 1]
[1, 2, 3][::-1] = [3, 2, 1]


## 3. Postorder là gì? Tính tay trước khi code

Định nghĩa đệ quy:

```text
postorder(node) = postorder(con_1) + postorder(con_2) + ... + postorder(con_k) + [node.val]
```

"Post" = **sau**: node cha được ghi **sau** toàn bộ con cháu.

Tính tay ví dụ 1:

```text
postorder(3) = postorder(5) + postorder(6) + [3] = [5, 6, 3]
postorder(2) = [2],  postorder(4) = [4]
postorder(1) = [5, 6, 3] + [2] + [4] + [1] = [5, 6, 3, 2, 4, 1]
```

So sánh với preorder (bài 589) của cùng cây: `[1, 3, 5, 6, 2, 4]`. Hãy để ý: **đảo ngược postorder** được `[1, 4, 2, 3, 6, 5]` – đó là "preorder nhưng duyệt con từ PHẢI sang TRÁI". Quan sát này là chìa khóa của Cách 2.

In [3]:
post = [5, 6, 3, 2, 4, 1]
print("postorder        :", post)
print("đảo ngược        :", post[::-1])
print("preorder thường  :", [1, 3, 5, 6, 2, 4])

postorder        : [5, 6, 3, 2, 4, 1]
đảo ngược        : [1, 4, 2, 3, 6, 5]
preorder thường  : [1, 3, 5, 6, 2, 4]


## 4. Cách 1 — Đệ quy

### 4.1. Ý tưởng
Hàm `dfs(node)`:
1. Base case: `node is None` → return.
2. Duyệt từng con: `for child in node.children: dfs(child)`.
3. **Sau** vòng for mới `result.append(node.val)`.

Chỉ khác preorder đúng **một dòng**: vị trí của `append`.

In [4]:
def postorder_recursive(root):
    result = []

    def dfs(node):
        if node is None:
            return
        for child in node.children:   # xong hết con trước
            dfs(child)
        result.append(node.val)       # rồi mới ghi cha (post)

    dfs(root)
    return result


print(postorder_recursive(build_nary(EX1)))   # [5, 6, 3, 2, 4, 1]
print(postorder_recursive(build_nary(EX2)))
print(postorder_recursive(None))

[5, 6, 3, 2, 4, 1]
[2, 6, 14, 11, 7, 3, 12, 8, 4, 13, 9, 10, 5, 1]
[]


### 4.2. Trace call stack
In thụt lề theo độ sâu: thấy rõ node chỉ được "ghi" khi **rời** hàm (sau khi mọi con đã xong).

In [5]:
def trace_postorder_recursive(root):
    result = []

    def dfs(node, depth):
        pad = "  " * depth
        print(f"{pad}vào dfs({node.val})")
        for child in node.children:
            dfs(child, depth + 1)
        result.append(node.val)
        print(f"{pad}rời dfs({node.val}) -> ghi {node.val}, result = {result}")

    if root:
        dfs(root, 0)
    return result

trace_postorder_recursive(build_nary(EX1))

vào dfs(1)
  vào dfs(3)
    vào dfs(5)
    rời dfs(5) -> ghi 5, result = [5]
    vào dfs(6)
    rời dfs(6) -> ghi 6, result = [5, 6]
  rời dfs(3) -> ghi 3, result = [5, 6, 3]
  vào dfs(2)
  rời dfs(2) -> ghi 2, result = [5, 6, 3, 2]
  vào dfs(4)
  rời dfs(4) -> ghi 4, result = [5, 6, 3, 2, 4]
rời dfs(1) -> ghi 1, result = [5, 6, 3, 2, 4, 1]


[5, 6, 3, 2, 4, 1]

### 4.3. Code nộp LeetCode — Cách 1

In [6]:
class Solution:
    def postorder(self, root: 'Node') -> List[int]:
        result = []

        def dfs(node):
            if node is None:
                return
            for child in node.children or []:
                dfs(child)
            result.append(node.val)

        dfs(root)
        return result


print(Solution().postorder(build_nary(EX2)))

[2, 6, 14, 11, 7, 3, 12, 8, 4, 13, 9, 10, 5, 1]


## 5. Nút thắt: đệ quy sâu và vì sao postorder lặp khó hơn preorder

**Nút thắt 1 – độ sâu:** Python mặc định giới hạn khoảng 1000 frame. Cây "chuỗi" rất sâu sẽ gây `RecursionError`.

**Nút thắt 2 – thời điểm ghi:** Với preorder lặp, ta pop node là ghi ngay. Nhưng với postorder, khi pop node lần đầu ta **chưa được ghi** – phải đợi các con xong. Một stack đơn giản không nhớ được "node này đã xử lý con chưa". Có 3 cách vượt qua:

| Cách | Mẹo |
|---|---|
| 2 | Không cần nhớ: làm "preorder phải→trái" rồi **đảo** kết quả |
| 3 | Nhớ bằng **cờ** `visited` đi kèm node |
| 4 | Nhớ bằng **chỉ số con kế tiếp** (mô phỏng đúng call stack) |

In [7]:
def make_chain(length):
    root = cur = Node(0)
    for v in range(1, length):
        nxt = Node(v)
        cur.children.append(nxt)
        cur = nxt
    return root

deep = make_chain(20000)
print("Giới hạn đệ quy:", sys.getrecursionlimit())
try:
    postorder_recursive(deep)
    print("Đệ quy chạy được")
except RecursionError as e:
    print("Đệ quy lỗi:", type(e).__name__)

Giới hạn đệ quy: 1000
Đệ quy lỗi: RecursionError


## 6. Cách 2 — Preorder "phải → trái" rồi đảo ngược

### 6.1. Vì sao đúng?
Postorder: `P(con_1) ... P(con_k), node`. Đảo toàn bộ: `node, rev(P(con_k)), ..., rev(P(con_1))`.
Áp dụng lặp lại cho từng cây con: `rev(P(x))` = "ghi x trước, rồi các con **từ phải sang trái**". Đó chính là một kiểu preorder với thứ tự con đảo ngược!

### 6.2. Giảng từng đoạn
- Stack lấy **cuối** ra trước; muốn con **phải nhất** ra trước → đẩy con theo thứ tự **xuôi** (`stack.extend(node.children)`, không `reversed`).
- Pop là ghi ngay vào `out`.
- Cuối cùng `out.reverse()`.

In [8]:
def postorder_reverse_trick(root):
    if root is None:
        return []
    stack = [root]
    out = []
    while stack:
        node = stack.pop()
        out.append(node.val)          # thứ tự: node, con phải ..., con trái
        stack.extend(node.children)   # đẩy xuôi -> con phải nhất ra trước
    out.reverse()                     # đảo -> đúng postorder
    return out


print(postorder_reverse_trick(build_nary(EX1)))
print(postorder_reverse_trick(build_nary(EX2)))
print("Cây chuỗi 20000 node, 3 giá trị cuối:", postorder_reverse_trick(deep)[-3:])

[5, 6, 3, 2, 4, 1]
[2, 6, 14, 11, 7, 3, 12, 8, 4, 13, 9, 10, 5, 1]
Cây chuỗi 20000 node, 3 giá trị cuối: [2, 1, 0]


### 6.3. TRACE cách 2 (ví dụ 1)

Stack ghi từ đáy → đỉnh.

| Bước | stack trước | pop | out sau | đẩy (xuôi) | stack sau |
|---:|---|:---:|---|---|---|
| 1 | `[1]` | 1 | `[1]` | 3, 2, 4 | `[3, 2, 4]` |
| 2 | `[3, 2, 4]` | 4 | `[1, 4]` | – | `[3, 2]` |
| 3 | `[3, 2]` | 2 | `[1, 4, 2]` | – | `[3]` |
| 4 | `[3]` | 3 | `[1, 4, 2, 3]` | 5, 6 | `[5, 6]` |
| 5 | `[5, 6]` | 6 | `[1, 4, 2, 3, 6]` | – | `[5]` |
| 6 | `[5]` | 5 | `[1, 4, 2, 3, 6, 5]` | – | `[]` |

Đảo `out` → `[5, 6, 3, 2, 4, 1]` ✔.

In [9]:
def trace_reverse_trick(root):
    stack, out, step = [root], [], 1
    while stack:
        before = [x.val for x in stack]
        node = stack.pop()
        out.append(node.val)
        stack.extend(node.children)
        print(f"Bước {step}: stack trước={before} | pop {node.val} | out={out} "
              f"| đẩy {[c.val for c in node.children]} | stack sau={[x.val for x in stack]}")
        step += 1
    out.reverse()
    print("Đảo out ->", out)
    return out

trace_reverse_trick(build_nary(EX1))

Bước 1: stack trước=[1] | pop 1 | out=[1] | đẩy [3, 2, 4] | stack sau=[3, 2, 4]
Bước 2: stack trước=[3, 2, 4] | pop 4 | out=[1, 4] | đẩy [] | stack sau=[3, 2]
Bước 3: stack trước=[3, 2] | pop 2 | out=[1, 4, 2] | đẩy [] | stack sau=[3]
Bước 4: stack trước=[3] | pop 3 | out=[1, 4, 2, 3] | đẩy [5, 6] | stack sau=[5, 6]
Bước 5: stack trước=[5, 6] | pop 6 | out=[1, 4, 2, 3, 6] | đẩy [] | stack sau=[5]
Bước 6: stack trước=[5] | pop 5 | out=[1, 4, 2, 3, 6, 5] | đẩy [] | stack sau=[]
Đảo out -> [5, 6, 3, 2, 4, 1]


[5, 6, 3, 2, 4, 1]

### 6.4. Code nộp LeetCode — Cách 2 (bản chính, ngắn nhất)

In [10]:
class Solution:
    def postorder(self, root: 'Node') -> List[int]:
        if root is None:
            return []
        stack, out = [root], []
        while stack:
            node = stack.pop()
            out.append(node.val)
            if node.children:
                stack.extend(node.children)
        return out[::-1]


sol = Solution()
assert sol.postorder(build_nary(EX1)) == [5, 6, 3, 2, 4, 1]
assert sol.postorder(build_nary(EX2)) == [2, 6, 14, 11, 7, 3, 12, 8, 4, 13, 9, 10, 5, 1]
assert sol.postorder(None) == []
print("Cách 2 PASS ví dụ đề")

Cách 2 PASS ví dụ đề


## 7. Cách 3 — Stack kèm cờ "đã mở con" (`visited`)

Cách 2 hơi "mẹo". Nếu bài yêu cầu **xử lý node đúng lúc** (ví dụ tính toán từ con lên cha) thì không đảo kết quả được. Ta cần một stack thật sự ghi đúng thứ tự.

Ý tưởng: mỗi phần tử stack là **tuple** `(node, visited)`:
- `visited = False`: lần đầu gặp → đẩy lại chính nó với `True`, rồi đẩy các con (ngược, để con trái ra trước).
- `visited = True`: các con đã được xử lý xong (vì chúng nằm trên nó trong stack) → **ghi** node.

**Tuple** là dãy giá trị không sửa được, viết `(a, b)`; có thể "mở gói" `node, visited = stack.pop()`.

In [11]:
pair = ("node_A", False)
name, flag = pair          # mở gói tuple
print(name, flag)

node_A False


In [12]:
def postorder_visited_flag(root):
    if root is None:
        return []
    stack = [(root, False)]
    result = []
    while stack:
        node, visited = stack.pop()
        if visited:
            result.append(node.val)                 # con đã xong -> ghi
        else:
            stack.append((node, True))              # hẹn quay lại
            for child in reversed(node.children):   # con trái ra trước
                stack.append((child, False))
    return result


print(postorder_visited_flag(build_nary(EX1)))
print(postorder_visited_flag(build_nary(EX2)) == postorder_recursive(build_nary(EX2)))

[5, 6, 3, 2, 4, 1]
True


### 7.1. TRACE cách 3
Ký hiệu `3*` nghĩa là `(3, True)` – "đã mở con, lần sau gặp là ghi".

In [13]:
def trace_visited_flag(root):
    stack, result, step = [(root, False)], [], 1
    fmt = lambda st: [f"{n.val}{'*' if v else ''}" for n, v in st]
    while stack:
        before = fmt(stack)
        node, visited = stack.pop()
        if visited:
            result.append(node.val)
            action = f"ghi {node.val}"
        else:
            stack.append((node, True))
            for child in reversed(node.children):
                stack.append((child, False))
            action = f"mở {node.val}, đẩy {node.val}* và các con"
        print(f"Bước {step:>2}: stack={before} -> {action} | result={result}")
        step += 1
    return result

trace_visited_flag(build_nary(EX1))

Bước  1: stack=['1'] -> mở 1, đẩy 1* và các con | result=[]
Bước  2: stack=['1*', '4', '2', '3'] -> mở 3, đẩy 3* và các con | result=[]
Bước  3: stack=['1*', '4', '2', '3*', '6', '5'] -> mở 5, đẩy 5* và các con | result=[]
Bước  4: stack=['1*', '4', '2', '3*', '6', '5*'] -> ghi 5 | result=[5]
Bước  5: stack=['1*', '4', '2', '3*', '6'] -> mở 6, đẩy 6* và các con | result=[5]
Bước  6: stack=['1*', '4', '2', '3*', '6*'] -> ghi 6 | result=[5, 6]
Bước  7: stack=['1*', '4', '2', '3*'] -> ghi 3 | result=[5, 6, 3]
Bước  8: stack=['1*', '4', '2'] -> mở 2, đẩy 2* và các con | result=[5, 6, 3]
Bước  9: stack=['1*', '4', '2*'] -> ghi 2 | result=[5, 6, 3, 2]
Bước 10: stack=['1*', '4'] -> mở 4, đẩy 4* và các con | result=[5, 6, 3, 2]
Bước 11: stack=['1*', '4*'] -> ghi 4 | result=[5, 6, 3, 2, 4]
Bước 12: stack=['1*'] -> ghi 1 | result=[5, 6, 3, 2, 4, 1]


[5, 6, 3, 2, 4, 1]

## 8. Cách 4 — Stack `[node, chỉ số con kế tiếp]` mô phỏng đúng đệ quy

Đây là cách "chuyển đệ quy thành vòng lặp" tổng quát nhất: mỗi phần tử stack giống một frame của call stack, nhớ "đang ở con thứ mấy".

- Đỉnh `[node, i]`: nếu `i < len(node.children)` → tăng `i`, đẩy con thứ `i` (= "gọi dfs(con)").
- Nếu hết con → pop và **ghi** `node.val` (= "hàm dfs(node) kết thúc").

Ưu điểm: stack chỉ cao `O(h)`, và mỗi node được ghi đúng lúc hàm đệ quy tương ứng kết thúc.

In [14]:
def postorder_index_stack(root):
    if root is None:
        return []
    stack = [[root, 0]]
    result = []
    while stack:
        node, i = stack[-1]
        if i < len(node.children):
            stack[-1][1] += 1
            stack.append([node.children[i], 0])
        else:
            stack.pop()
            result.append(node.val)
    return result


print(postorder_index_stack(build_nary(EX1)))
print(postorder_index_stack(build_nary(EX2)))

[5, 6, 3, 2, 4, 1]
[2, 6, 14, 11, 7, 3, 12, 8, 4, 13, 9, 10, 5, 1]


In [15]:
def trace_index_stack(root):
    stack, result = [[root, 0]], []
    while stack:
        node, i = stack[-1]
        view = [(n.val, k) for n, k in stack]
        if i < len(node.children):
            stack[-1][1] += 1
            stack.append([node.children[i], 0])
            print(f"stack={view} -> đi vào con #{i} của {node.val}")
        else:
            stack.pop()
            result.append(node.val)
            print(f"stack={view} -> {node.val} hết con, ghi -> result={result}")
    return result

trace_index_stack(build_nary(EX1))

stack=[(1, 0)] -> đi vào con #0 của 1
stack=[(1, 1), (3, 0)] -> đi vào con #0 của 3
stack=[(1, 1), (3, 1), (5, 0)] -> 5 hết con, ghi -> result=[5]
stack=[(1, 1), (3, 1)] -> đi vào con #1 của 3
stack=[(1, 1), (3, 2), (6, 0)] -> 6 hết con, ghi -> result=[5, 6]
stack=[(1, 1), (3, 2)] -> 3 hết con, ghi -> result=[5, 6, 3]
stack=[(1, 1)] -> đi vào con #1 của 1
stack=[(1, 2), (2, 0)] -> 2 hết con, ghi -> result=[5, 6, 3, 2]
stack=[(1, 2)] -> đi vào con #2 của 1
stack=[(1, 3), (4, 0)] -> 4 hết con, ghi -> result=[5, 6, 3, 2, 4]
stack=[(1, 3)] -> 1 hết con, ghi -> result=[5, 6, 3, 2, 4, 1]


[5, 6, 3, 2, 4, 1]

## 9. Vì sao đúng? (invariant)

**Cách 3 (cờ visited).** Invariant: một phần tử `(x, True)` trong stack luôn nằm **dưới** toàn bộ các phần tử `(con cháu của x, ...)` được đẩy sau nó. Vì stack là LIFO, `(x, True)` chỉ được pop khi mọi con cháu của `x` đã được pop và ghi → `x` được ghi sau các con cháu (đúng "post"). Các con được đẩy ngược nên con trái được xử lý xong trước con phải → đúng thứ tự anh em.

**Cách 2 (đảo).** `out` là preorder với thứ tự con đảo (`node`, rồi cây con phải nhất, …). Theo mục 6.1, đảo ngược nó cho đúng `P(con_1) ... P(con_k) node` – chứng minh bằng quy nạp theo chiều cao.

**Cách 4 (chỉ số).** Mỗi phần tử stack đúng bằng một frame của `dfs`; `i` là vị trí của vòng `for` trong frame. Pop = hàm kết thúc = thời điểm đệ quy ghi `node.val`. Hai chương trình có cùng chuỗi sự kiện nên cùng kết quả.

**Không trùng, không sót:** mỗi node được đẩy vào stack (với cờ `False` / chỉ số 0) đúng một lần – khi cha "mở" nó.

## 10. Time / Space Complexity và bảng so sánh

`n` = số node, `h` = chiều cao.

| Cách | Time | Space phụ | Ưu điểm | Nhược điểm |
|---|---|---|---|---|
| 1. Đệ quy | `O(n)` | `O(h)` call stack | Ngắn, sát định nghĩa | Có thể `RecursionError` |
| 2. Preorder phải→trái + đảo | `O(n)` | `O(n)` | Ngắn nhất bản lặp | Chỉ ra *thứ tự*, không xử lý node đúng lúc |
| 3. Stack + cờ visited | `O(n)` | `O(n)` | Ghi đúng thứ tự, dễ nhớ | Mỗi node vào stack 2 lần |
| 4. Stack `(node, i)` | `O(n)` | `O(h)` | Mô phỏng chính xác đệ quy | Code dài hơn chút |

Time `O(n)` vì mỗi node vào/ra stack hằng số lần và mỗi cạnh cha→con được xét một lần (tổng `n - 1` cạnh). Phép đảo list ở cách 2 thêm `O(n)` – vẫn `O(n)`.

## 11. Bộ test: mẫu đề, biên, đối chiếu ngẫu nhiên

In [16]:
methods = {
    "recursive": postorder_recursive,
    "reverse_trick": postorder_reverse_trick,
    "visited_flag": postorder_visited_flag,
    "index_stack": postorder_index_stack,
}
star = Node(0, [Node(i) for i in range(1, 6)])

cases = [
    ("đề ví dụ 1", build_nary(EX1), [5, 6, 3, 2, 4, 1]),
    ("đề ví dụ 2", build_nary(EX2), [2, 6, 14, 11, 7, 3, 12, 8, 4, 13, 9, 10, 5, 1]),
    ("cây rỗng", None, []),
    ("1 node", build_nary([42]), [42]),
    ("hình sao", star, [1, 2, 3, 4, 5, 0]),
    ("chuỗi 5 node", make_chain(5), [4, 3, 2, 1, 0]),
]
for name, root, expected in cases:
    for mname, f in methods.items():
        got = f(root)
        assert got == expected, (name, mname, got)
    print(f"{name:<14} PASS cả 4 cách -> {expected}")

đề ví dụ 1     PASS cả 4 cách -> [5, 6, 3, 2, 4, 1]
đề ví dụ 2     PASS cả 4 cách -> [2, 6, 14, 11, 7, 3, 12, 8, 4, 13, 9, 10, 5, 1]
cây rỗng       PASS cả 4 cách -> []
1 node         PASS cả 4 cách -> [42]
hình sao       PASS cả 4 cách -> [1, 2, 3, 4, 5, 0]
chuỗi 5 node   PASS cả 4 cách -> [4, 3, 2, 1, 0]


In [17]:
def random_nary(n, seed):
    rng = random.Random(seed)
    nodes = [Node(rng.randint(0, 50))]
    for _ in range(n - 1):
        child = Node(rng.randint(0, 50))
        rng.choice(nodes).children.append(child)
        nodes.append(child)
    return nodes[0]

for seed in range(300):
    root = random_nary(1 + seed % 40, seed)
    expected = postorder_recursive(root)
    assert expected[-1] == root.val          # tính chất: root luôn cuối
    for mname, f in methods.items():
        assert f(root) == expected, (seed, mname)
print("300 cây ngẫu nhiên: 4 cách khớp nhau, root luôn đứng cuối — PASS")

300 cây ngẫu nhiên: 4 cách khớp nhau, root luôn đứng cuối — PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Ở cách 2 vẫn đẩy con bằng `reversed`
Nếu đẩy ngược (như preorder) rồi đảo kết quả, thứ tự anh em sẽ bị lật.

### Lỗi 2: Ghi node ngay khi pop lần đầu (quên cờ visited)
Thành preorder chứ không phải postorder.

### Lỗi 3: `return out.reverse()`
`list.reverse()` đảo tại chỗ và **trả về `None`**. Phải `out.reverse(); return out` hoặc `return out[::-1]`.

In [18]:
def post_wrong_reversed(root):
    stack, out = [root], []
    while stack:
        node = stack.pop()
        out.append(node.val)
        stack.extend(reversed(node.children))   # SAI cho mẹo đảo
    return out[::-1]

def post_wrong_return(root):
    stack, out = [root], []
    while stack:
        node = stack.pop()
        out.append(node.val)
        stack.extend(node.children)
    return out.reverse()                        # SAI: trả về None

t = build_nary(EX1)
print("Đẩy reversed rồi đảo :", post_wrong_reversed(t), "(đúng là [5, 6, 3, 2, 4, 1])")
print("return out.reverse() :", post_wrong_return(t))

Đẩy reversed rồi đảo : [4, 2, 6, 5, 3, 1] (đúng là [5, 6, 3, 2, 4, 1])
return out.reverse() : None


### Lỗi 4: Ở cách 4 quên tăng chỉ số `i`
`stack[-1][1] += 1` bị thiếu → con thứ 0 bị đẩy mãi → vòng lặp vô hạn. (Không chạy demo để tránh treo notebook.)

### Lỗi 5: Dùng tuple cho cách 4
`stack[-1][1] += 1` với tuple sẽ lỗi `TypeError` vì tuple không sửa được. Dùng list `[node, i]`.

In [19]:
frame = (Node(1), 0)
try:
    frame[1] += 1
except TypeError as e:
    print("TypeError:", e)
frame = [Node(1), 0]
frame[1] += 1
print("Dùng list thì sửa được:", frame)

TypeError: 'tuple' object does not support item assignment
Dùng list thì sửa được: [Node(1), 1]


## 13. Mẹo nhận dạng pattern và biến thể

**Khi nào cần postorder?** Khi giá trị của node **phụ thuộc vào kết quả của các con**: chiều cao cây, kích thước cây con, tổng cây con, xóa cây (xóa con trước cha), tính biểu thức (toán hạng trước, toán tử sau).

**Ba câu thần chú:**
- Preorder lặp: *pop – ghi – đẩy ngược*.
- Postorder mẹo: *pop – ghi – đẩy xuôi – cuối cùng đảo*.
- Postorder "chuẩn": *cờ visited* hoặc *(node, i)*.

**Biến thể liên quan:** 145 (Binary Tree Postorder), 589 (Preorder N-ary), 559 (Max Depth N-ary – bản chất là postorder), 429 (Level order N-ary).

Demo: tính **tổng cây con** cho mọi node – một bài cần postorder thật sự (cha cần tổng của con).

In [20]:
def subtree_sums(root):
    """Trả về dict {giá trị node: tổng cây con} bằng stack + cờ visited."""
    total = {}
    stack = [(root, False)]
    while stack:
        node, visited = stack.pop()
        if visited:
            total[node] = node.val + sum(total[c] for c in node.children)
        else:
            stack.append((node, True))
            for c in node.children:
                stack.append((c, False))
    return {node.val: s for node, s in total.items()}

print(subtree_sums(build_nary(EX1)))   # 3: 3+5+6=14, 1: 14+2+4+1=21

{4: 4, 2: 2, 6: 6, 5: 5, 3: 14, 1: 21}


## 14. Bài tập (TODO – không gọi khi chưa làm xong)

**Bài 1 — Tính tay.** Viết postorder của cây `[10,null,20,30,null,40,50,null,60]`.

**Bài 2 — Đoán output.** Với cây ví dụ 1, hàm cách 2 trước khi đảo (`out`) chứa gì?

**Bài 3 — Viết lại postorder** bằng cờ `visited` (không nhìn code).

**Bài 4 — Chiều cao cây N-ary** bằng postorder lặp (không đệ quy).

**Bài 5 — Đếm số lá** (node không có con).

**Bài 6 — Mini challenge:** cho cây N-ary, trả về list `(val, size)` theo **postorder**, trong đó `size` là số node của cây con gốc tại node đó.

In [21]:
answer_ex1_by_hand = None     # TODO: list postorder bạn tính tay
answer_ex2_out_before_reverse = None   # TODO


def postorder_flag_exercise(root):
    # TODO 1: cây rỗng -> []
    # TODO 2: stack chứa (node, visited)
    # TODO 3: visited -> ghi; chưa -> đẩy lại (node, True) rồi đẩy con ngược
    pass


def height_iterative(root):
    # TODO: postorder bằng stack, lưu height[node] = 1 + max height các con
    pass

# Kiểm tra sau khi làm:
# assert postorder_flag_exercise(build_nary(EX1)) == [5, 6, 3, 2, 4, 1]
# assert height_iterative(build_nary(EX2)) == 5

In [22]:
def count_leaves(root):
    # TODO: lá là node có children rỗng
    pass


def post_sizes(root):
    # TODO: trả về list (val, size) theo postorder
    pass

# Kiểm tra sau khi làm:
# assert count_leaves(build_nary(EX1)) == 4
# assert post_sizes(build_nary(EX1)) == [(5, 1), (6, 1), (3, 3), (2, 1), (4, 1), (1, 6)]

## 15. Gợi ý 3 mức

**Bài 1–2**
1. Hướng: postorder = con trước, cha sau; `out` của cách 2 là preorder duyệt con phải → trái.
2. Kiểm tra: phần tử cuối postorder phải là root `10`.
3. Gần đáp án: bắt đầu bằng `40`.

**Bài 3**
1. Hướng: mỗi node gặp 2 lần: lần 1 "mở", lần 2 "ghi".
2. Cú pháp: `node, visited = stack.pop()`.
3. Gần code: xem bảng trace mục 7.1, viết lại hai nhánh `if visited / else`.

**Bài 4**
1. Hướng: chiều cao node = 1 + max chiều cao các con → cần con xong trước (postorder).
2. Cú pháp: dict `height = {}`; `max((height[c] for c in node.children), default=0)`.
3. Gần code: dùng khung cờ visited; ở nhánh `visited` tính `height[node]`; trả `height[root]`.

**Bài 5**
1. Hướng: duyệt kiểu nào cũng được, đếm node có `not node.children`.
2. Cú pháp: `sum(1 for ...)` hoặc biến đếm.
3. Gần code: dùng lại vòng stack của cách 2.

**Bài 6**
1. Hướng: size = 1 + tổng size các con, ghi theo đúng thứ tự postorder.
2. Cú pháp: dict `size = {}` và list `out` append `(node.val, size[node])` ở nhánh ghi.
3. Gần code: khung cờ visited, nhánh `visited` tính size và append.

## 16. Đáp án tham khảo (chạy kiểm tra)

In [23]:
ex_tree = build_nary([10, None, 20, 30, None, 40, 50, None, 60])
show_nary(ex_tree)
ans1 = [40, 50, 20, 60, 30, 10]
ans2 = [1, 4, 2, 3, 6, 5]


def ans_postorder_flag(root):
    if root is None:
        return []
    stack, result = [(root, False)], []
    while stack:
        node, visited = stack.pop()
        if visited:
            result.append(node.val)
        else:
            stack.append((node, True))
            for child in reversed(node.children):
                stack.append((child, False))
    return result


def ans_height_iterative(root):
    if root is None:
        return 0
    height = {}
    stack = [(root, False)]
    while stack:
        node, visited = stack.pop()
        if visited:
            height[node] = 1 + max((height[c] for c in node.children), default=0)
        else:
            stack.append((node, True))
            for c in node.children:
                stack.append((c, False))
    return height[root]


def ans_count_leaves(root):
    if root is None:
        return 0
    stack, cnt = [root], 0
    while stack:
        node = stack.pop()
        if not node.children:
            cnt += 1
        stack.extend(node.children)
    return cnt


def ans_post_sizes(root):
    if root is None:
        return []
    size, out = {}, []
    stack = [(root, False)]
    while stack:
        node, visited = stack.pop()
        if visited:
            size[node] = 1 + sum(size[c] for c in node.children)
            out.append((node.val, size[node]))
        else:
            stack.append((node, True))
            for c in reversed(node.children):
                stack.append((c, False))
    return out


assert ans_postorder_flag(ex_tree) == ans1
assert postorder_reverse_trick(ex_tree) == ans1
out_before = postorder_reverse_trick(build_nary(EX1))[::-1]
assert out_before == ans2
assert ans_postorder_flag(build_nary(EX1)) == [5, 6, 3, 2, 4, 1]
assert ans_height_iterative(build_nary(EX2)) == 5
assert ans_height_iterative(deep) == 20000
assert ans_count_leaves(build_nary(EX1)) == 4
assert ans_post_sizes(build_nary(EX1)) == [(5, 1), (6, 1), (3, 3), (2, 1), (4, 1), (1, 6)]
print("Bài 1:", ans1, "| Bài 2:", ans2)
print("Đáp án: tất cả test PASS")

10
    20
        40
        50
    30
        60
Bài 1: [40, 50, 20, 60, 30, 10] | Bài 2: [1, 4, 2, 3, 6, 5]
Đáp án: tất cả test PASS


## 17. Template nhớ nhanh và checklist submit

```python
class Solution:
    def postorder(self, root: 'Node') -> List[int]:
        if root is None:
            return []
        stack, out = [root], []
        while stack:
            node = stack.pop()
            out.append(node.val)
            if node.children:
                stack.extend(node.children)   # đẩy XUÔI
        return out[::-1]                      # đảo cuối cùng
```

Khi cần xử lý node đúng lúc (cha dùng kết quả con) → dùng **cờ visited**:

```python
stack = [(root, False)]
while stack:
    node, visited = stack.pop()
    if visited:
        ...                       # mọi con đã xong
    else:
        stack.append((node, True))
        for child in reversed(node.children):
            stack.append((child, False))
```

### Checklist
- [ ] Cây rỗng trả `[]`?
- [ ] Root đứng **cuối** kết quả?
- [ ] Cách đảo: đẩy con **xuôi**, đảo bằng `[::-1]` (không `return out.reverse()`)?
- [ ] Cách cờ: đẩy `(node, True)` **trước** khi đẩy các con?
- [ ] Xử lý `children` có thể là `None` trên LeetCode (`if node.children`)?
- [ ] Đã thử cây rỗng, 1 node, chuỗi sâu, hình sao?